<a href="https://colab.research.google.com/github/roughhawkbit/digi-inno-road-prod/blob/main/analysis/Analyse_BART_zero_shot_results.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [ ]:
import matplotlib
import numpy
import os
import pandas
import sys

In [ ]:
if IN_COLAB:
  dirpath = '/content/digi-inno-road-prod'
  if not os.path.isdir(dirpath):
    # TODO git pull
    !git clone https://github.com/roughhawkbit/digi-inno-road-prod.git
  sys.path.insert(0,dirpath)
else:
  module_path = os.path.abspath(os.path.join('..'))
  if not module_path in sys.path:
      sys.path.insert(0, module_path)

In [ ]:
from innoprod.sheet_tools import get_sheet_dfs
from innoprod.plotting_tools import rand_jitter

# Read in data

In [ ]:
all_results_df = get_sheet_dfs(
    sheet_id="1_ycDOfNq7khbzgdgF80qhHWmWpHC_9TGIwdDkkAvfvY",
    ranges={"Results": "Sheet1!A1:H3301"}
  )['Results']
# all_results_df

In [ ]:
col_types = {
    'Client ID': 'str',
    'Current DRS': pandas.Int64Dtype(),
    'Core': 'str',
    'Category': 'str',
    'Question': 'str',
    'Prediction': 'int',
    'Probability': 'float',
    'Confidence': 'float'
}

all_results_df['Current DRS'] = all_results_df['Current DRS'].replace(to_replace='', value=numpy.nan)

for col, ty in col_types.items():
  all_results_df[col] = all_results_df[col].astype(ty)


# Explore results

In [ ]:
all_results_df['Prediction'].value_counts().sort_index()

In [ ]:
bart_output_cols = ['Prediction',	'Probability',	'Confidence']

In [ ]:
all_results_df[['Question']+bart_output_cols].groupby('Question').mean()

In [ ]:
client_weighted_average = all_results_df.groupby('Client ID')[['Client ID', 'Prediction', 'Confidence']].apply(
    lambda x: numpy.average(x['Prediction'], weights=x['Confidence'])
  )

per_client_results = client_weighted_average.to_frame().reset_index()
per_client_results.columns = ['Client ID', 'Weighted Average']
per_client_results = pandas.merge(
    per_client_results,
    all_results_df[['Client ID', 'Current DRS']].drop_duplicates(),
    on="Client ID",
    how="left"
  )

per_client_results

In [ ]:
fig, ax = matplotlib.pyplot.subplots(figsize=(6,6))
ax.hist(client_weighted_average.to_numpy())
ax.set_xlabel('Average W/C/C score, weighted by prediction confidence')
ax.set_ylabel('Number of firms')

In [ ]:
fig, ax = matplotlib.pyplot.subplots(figsize=(6,6))
X = per_client_results['Current DRS'].to_numpy().reshape(-1,1)
Y = per_client_results['Weighted Average'].to_numpy().reshape(-1,1)
ax.scatter(rand_jitter(X), Y)
ax.set_xlabel('Current DRS')
ax.set_ylabel('Average W/C/C score, weighted by prediction confidence')